## API pagination

API pagination refers to a technique used in API design and development to retrieve large data sets in a structured and manageable manner. When an API endpoint returns a large amount of data, pagination allows the data to be divided into smaller, more manageable chunks or pages. 

#### Offset Pagination

Offset pagination is a technique where a client specifies the number of records to skip (offset) and the number of records to retrieve (limit) in order to fetch a subset of data from a larger dataset.

offset=0   → 1–100  
offset=100 → 101–200  
offset=200 → 201–300  

In [2]:
import requests

url = "https://dummyjson.com/products"

limit = 10
skip = 0 # offset

all_data = []

while True:
    params = {
        "limit": limit,
        "skip": skip
    }

    response = requests.get(url, params=params)
    data = response.json()

    products = data["products"]

    if not products:
        break

    all_data.extend(products)

    print(f"Fetched {len(products)} records, skip={skip}")

    skip = skip + limit

print("Total:", len(all_data))

Fetched 10 records, skip=0
Fetched 10 records, skip=10
Fetched 10 records, skip=20
Fetched 10 records, skip=30
Fetched 10 records, skip=40
Fetched 10 records, skip=50
Fetched 10 records, skip=60
Fetched 10 records, skip=70
Fetched 10 records, skip=80
Fetched 10 records, skip=90
Fetched 10 records, skip=100
Fetched 10 records, skip=110
Fetched 10 records, skip=120
Fetched 10 records, skip=130
Fetched 10 records, skip=140
Fetched 10 records, skip=150
Fetched 10 records, skip=160
Fetched 10 records, skip=170
Fetched 10 records, skip=180
Fetched 4 records, skip=190
Total: 194


<br> Data order could change once New records inserted so sorting is necessary
<br> **Offset pagination must always be combined with sorting on a stable column (id or timestamp)**

 *params = {
 <br>       "limit": limit,
 <br>       "offset": offset,
 <br>       "sort": "id",             
       "order": "asc"             # asc / desc
    }*


### Advantage of using Offset pagination
<br> Good for small dataset
<br> Simple Implementation

### Limitation
<br> performance issues: As it scans skipped rows
<br> High Database Load
<br> not recommended for real time data

#### Cursor Pagination

Cursor pagination is a technique where the client retrieves the next set of records based on a reference value (cursor) from the last fetched record, instead of skipping rows.

Cursor is typically:

Primary key (id)
<br> Timestamp (created_at)

| Type           | How cursor looks          |
| -------------- | ------------------------- |
| 1. ID-based    | `cursor = last_id`        |
| 2. Token-based | `cursor = encoded_string` |
| 3. URL-based   | `cursor = next_url`       |


**ID based**

ID-based cursor pagination is a method of fetching data in chunks where the last record’s unique ID from the previous response is used as a cursor to retrieve the next set of records.

In [18]:


url = "https://dummyjson.com/products"

limit = 5
cursor = 0   # start before first id

while True:
    response = requests.get(url)
    data = response.json()

    products = data["products"]

    # applying cursor logic to only understand the working, 
    #in real case we have to capture the cursor_details from response and use it in next API hit
    
    filtered = [p for p in products if p["id"] > cursor][:limit]

    if not filtered:
        break
        
    print("\n Cursor :", cursor)
    print("\nData:\n")

    for p in filtered:
        print(p["id"], p["title"])

    # IMPORTANT: update cursor
    cursor = filtered[-1]["id"]


 Cursor : 0

Data:

1 Essence Mascara Lash Princess
2 Eyeshadow Palette with Mirror
3 Powder Canister
4 Red Lipstick
5 Red Nail Polish

 Cursor : 5

Data:

6 Calvin Klein CK One
7 Chanel Coco Noir Eau De
8 Dior J'adore
9 Dolce Shine Eau de
10 Gucci Bloom Eau de

 Cursor : 10

Data:

11 Annibale Colombo Bed
12 Annibale Colombo Sofa
13 Bedside Table African Cherry
14 Knoll Saarinen Executive Conference Chair
15 Wooden Bathroom Sink With Mirror

 Cursor : 15

Data:

16 Apple
17 Beef Steak
18 Cat Food
19 Chicken Meat
20 Cooking Oil

 Cursor : 20

Data:

21 Cucumber
22 Dog Food
23 Eggs
24 Fish Steak
25 Green Bell Pepper

 Cursor : 25

Data:

26 Green Chili Pepper
27 Honey Jar
28 Ice Cream
29 Juice
30 Kiwi


<br> 

**Token Based**

Token-based cursor pagination is a method of retrieving data in chunks where the server returns an encoded token (cursor) representing the current position, and this token is used in the next request to fetch the subsequent set of records.

<br> 

**URL-Based Cursor Pagination**

URL-based cursor pagination is a method where the API returns the next page URL, and the client simply uses that URL to fetch the next set of data.

In [21]:
import requests

url = "https://pokeapi.co/api/v2/pokemon"

while url:
    
    params = {'limit': 5}
    response = requests.get(url, params = params)
    data = response.json()

    print("\nData:\n")

    for p in data["results"]:
        print(p["name"])

    # cursor = next URL
    url = data["next"]
    print("\n url: ", url)
    
    # break condition
    
    if not url:
        print("\nNo more data. Stopping...")
        break



Data:

bulbasaur
ivysaur
venusaur
charmander
charmeleon

 url:  https://pokeapi.co/api/v2/pokemon?offset=5&limit=5

Data:

charizard
squirtle
wartortle
blastoise
caterpie

 url:  https://pokeapi.co/api/v2/pokemon?offset=10&limit=5

Data:

metapod
butterfree
weedle
kakuna
beedrill

 url:  https://pokeapi.co/api/v2/pokemon?offset=15&limit=5

Data:

pidgey
pidgeotto
pidgeot
rattata
raticate

 url:  https://pokeapi.co/api/v2/pokemon?offset=20&limit=5

Data:

spearow
fearow
ekans
arbok
pikachu

 url:  https://pokeapi.co/api/v2/pokemon?offset=25&limit=5

Data:

raichu
sandshrew
sandslash
nidoran-f
nidorina

 url:  https://pokeapi.co/api/v2/pokemon?offset=30&limit=5

Data:

nidoqueen
nidoran-m
nidorino
nidoking
clefairy

 url:  https://pokeapi.co/api/v2/pokemon?offset=35&limit=5

Data:

clefable
vulpix
ninetales
jigglypuff
wigglytuff

 url:  https://pokeapi.co/api/v2/pokemon?offset=40&limit=5

Data:

zubat
golbat
oddish
gloom
vileplume

 url:  https://pokeapi.co/api/v2/pokemon?offset=45&limi

**Page Based Pagination**

Page-based pagination is a method where data is divided into numbered pages, and the client requests a specific page using a page number and page size.

#### Keyset pagination

Keyset pagination retrieves the next set of results using the last record’s sorted key values instead of OFFSET, ensuring high performance and consistency.

| Aspect                     | Cursor Pagination                          | Keyset Pagination                              |
| -------------------------- | ------------------------------------------ | ---------------------------------------------- |
| **Definition**    | API gives you a pointer → you send it back | You define position using actual column values |
| **Core Idea**              | Follow the cursor provided by API          | Fetch data based on last row’s sorted values   |
| **How it works** | “Here is next cursor → use it”             | “Give me data after this position”             |
| **Example (API Response)** | `next_cursor = "abc123"`                   | `(created_at, id) = ('2025-01-01', 100)`       |
| **Next Request Example**   | `GET /api?cursor=abc123`                   | `WHERE (created_at, id) > ('2025-01-01', 100)` |
| **Who controls logic**     | API controls everything                    | Developer controls query                    |


| Operator | Meaning               | SQL Equivalent |
| -------- | --------------------- | -------------- |
| `$lt`    | less than             | `<`            |
| `$gt`    | greater than          | `>`            |
| `$lte`   | less than or equal    | `<=`           |
| `$gte`   | greater than or equal | `>=`           |


In [6]:
import requests  

url = "https://api.spacexdata.com/v4/launches/query"
=
limit = 3  

last_date = None  

while True:  

    # Step 1: build query (filter condition)
    query = {}

    # Apply keyset condition only after first iteration
    if last_date:
        
        # This means: give records where date_utc < last_date
        query["date_utc"] = {"$lt": last_date}

    # Step 2: prepare payload for API
    payload = {
        "query": query,  # filter condition
        "options": {
            "limit": limit,  # how many records to fetch
            "sort": {"date_utc": -1}  # sort by date (latest first)
        }
    }

    # Step 3: call API using POST request
    response = requests.post(url, json=payload)

    # Extract actual data (API returns data inside "docs")
    data = response.json()["docs"]

    # Step 4: break condition (stop when no more data)
    if not data:
        break

    print("\nData:\n")

    # Step 5: print results
    for d in data:
        print(d["name"], d["date_utc"])

    # Step 6: update cursor
    # Take last record's date to fetch next set
    last_date = data[-1]["date_utc"]


Data:

SWOT 2022-12-05T00:00:00.000Z
TTL-1 2022-12-01T00:00:00.000Z
WorldView Legion 1 & 2 2022-12-01T00:00:00.000Z
2 2022-12-01T00:00:00.000Z

Data:

ispace Mission 1 & Rashid 2022-11-22T00:00:00.000Z
CRS-26 2022-11-18T22:00:00.000Z
Eutelsat 10B 2022-11-15T00:00:00.000Z
2 2022-11-15T00:00:00.000Z

Data:

Galaxy 31 (23R) & 32 (17R) 2022-11-08T00:00:00.000Z
Hotbird 13G 2022-11-03T03:24:00.000Z
USSF-44 2022-11-01T13:41:00.000Z
2 2022-11-01T13:41:00.000Z

Data:

SES-18 & SES-19 2022-11-01T00:00:00.000Z
Starlink 4-37 (v1.5) 2022-11-01T00:00:00.000Z
O3b mPower 1,2 2022-11-01T00:00:00.000Z
2 2022-11-01T00:00:00.000Z

Data:

Starlink 4-36 (v1.5) 2022-10-20T14:50:00.000Z
Hotbird 13F 2022-10-15T05:22:00.000Z
Galaxy 33 (15R) & 34 (12R) 2022-10-08T23:05:00.000Z
2 2022-10-08T23:05:00.000Z

Data:

Crew-5 2022-10-05T16:00:00.000Z
Starlink 4-35 (v1.5) 2022-09-24T23:30:00.000Z
Starlink 4-34 (v1.5) 2022-09-17T01:05:00.000Z
2 2022-09-17T01:05:00.000Z

Data:

Starlink 4-2 (v1.5) & Blue Walker 3 2022-09-

| Type                              | Performance                         | Where to Use                  | Why                                |
| --------------------------------- | ----------------------------------- | ----------------------------- | ---------------------------------- |
| **Page-Based Pagination**         | ❌ Poor for large data               | UI (websites, admin panels)   | Easy for users to jump to any page |
| **Offset-Based Pagination**       | ❌ Poor (degrades with large OFFSET) | Small APIs, dashboards        | Simple to implement                |
| **ID-Based Cursor Pagination**    | ✅ Good (uses index)                 | APIs, ETL pipelines           | Fast and consistent                |
| **Token-Based Cursor Pagination** | ✅ Good                              | Secure APIs (SaaS platforms)  | Hides internal logic               |
| **URL-Based Cursor Pagination**   | ✅ Good                              | Public APIs                   | Simplest for clients               |
| **Keyset Pagination (Advanced)**  | Best (very fast)                 | High-scale systems, databases | Handles large data efficiently     |
